<a href="https://colab.research.google.com/github/Maclenn77/big-data-team/blob/main/proyecto-2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Proyecto Etapa 2. Selección de técnica de muestreo para la construcción de muestra inicial**

**Equipo 04**

**Daniela Montserrat Enríquez Ballesteros (A01797865)

**Humberto Alejandro Espinola Gonzalez (A01840066)

**Juan Paulo Pérez Tejada Ladrón de Guevara (A01797972)

**Mónica Raquel Saucedo Alcalá (A01797710)

## Inicialización y Carga de Datos

In [ ]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://dlcdn.apache.org/spark/spark-3.5.8/spark-3.5.8-bin-hadoop3.tgz
!tar -xzf spark-3.5.8-bin-hadoop3.tgz
!pip install -q findspark

In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.8-bin-hadoop3"

## Creación de Variables de Caracterización

In [ ]:
from pyspark.sql.functions import col, hour, dayofweek, when

df_augmented = df.withColumn("hour_of_day", hour(col("tpep_pickup_datetime")))
df_augmented = df_augmented.withColumn("day_of_week", dayofweek(col("tpep_pickup_datetime")))

# fines de semana
df_augmented = df_augmented.withColumn(
    "is_weekend",
    when(col("day_of_week").isin([1, 7]), True).otherwise(False)
)

#horas pico
df_augmented = df_augmented.withColumn(
    "is_rush_hour",
    when(col("hour_of_day").isin([7, 8, 9, 17, 18, 19]), True).otherwise(False)
)

## Extracción de Submuestras por Partición

In [ ]:
particion_1 = df_augmented.filter((col("is_weekend") == False) & (col("is_rush_hour") == True))
print(f"Total de registros en Partición 1: {particion_1.count()}")

In [ ]:
particion_2 = df_augmented.filter((col("is_weekend") == False) & (col("is_rush_hour") == False))
print(f"Total de registros en Partición 2: {particion_2.count()}")

In [ ]:
particion_3 = df_augmented.filter((col("is_weekend") == True) & (col("is_rush_hour") == True))
print(f"Total de registros en Partición 3: {particion_3.count()}")

In [ ]:
particion_4 = df_augmented.filter((col("is_weekend") == True) & (col("is_rush_hour") == False))
print(f"Total de registros en Partición 4: {particion_4.count()}")

In [ ]:
spark.stop()